In [1]:
import os

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
#load the document
loader = Docx2txtLoader('Ideology.docx')
document = loader.load()

In [10]:
#text splitt and chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size =500,
    chunk_overlap = 50
)
chunks = text_splitter.split_documents(document)

In [11]:
print('number of chunks: ',len(chunks))

number of chunks:  58


In [12]:
# embedding

embedding = HuggingFaceEmbeddings(
    model = 'sentence-transformers/all-MiniLM-L6-v2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
#creating vector database
vector_db = FAISS.from_documents(
    chunks,
    embedding
)

In [14]:
# creating retriver
retriver = vector_db.as_retriever(
    search_kwargd = {'k':5}
)

In [16]:
#prompt

prompt = ChatPromptTemplate.from_template(
    '''
    Provide the answer if it is in teh context,
    if the question is not in the context say: 'QUESTION OUT OF CONTEXT'

    Context
    {context}

    Question
    {question}
    
'''
)


In [17]:
# creating model
model = ChatGoogleGenerativeAI(
    model = 'gemini-3.6-flash',
    temperature = 0
)

In [18]:
# user query
question = input('Enter your question')

In [19]:
#retrive relevant info
retrived_data = retriver.invoke(question)

In [21]:
# context

context = '\n\n'.join(
    pages.page_content
    for pages in retrived_data
)

In [23]:
final_prompt = prompt.invoke({
    'context':context,
    'question':question
})

In [24]:
#generate the answrr
response = model.invoke(final_prompt)

e:\GEN-AI-PROJECTS\genai-env\Lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


In [25]:
print(response.text())

Based on the provided context, the role of ideology in foreign policy includes:

* **Influencing Goals and Means:** Ideology influences the choice of the goals and objectives of national interest, as well as the means used to secure those goals.
* **Policy Implementation and Defense:** It is used in foreign policy-making and implementation, where each foreign policy uses particular ideologies as "ideological weapons of defence."
* **Justification and Criticism:** Nations use ideology to explain, justify, and defend their own policies and actions, as well as to criticize and reject the policies of other nations.
* **Hiding Real Intentions:** Ideologies act as "cloaks" used by nations to hide their real intentions, which include maintaining and increasing their power in international relations.
